# Prática — Feature Engineering (Aula 3)
## Transformando as bases de eventos em uma tabela de modelagem

**O que vocês já têm:**
- O cadastro de colaboradores (`FtFuncionarioRH_amostra.csv`, Aula 2)
- O alvo já derivado (`alvo_derivado.csv`) — `pediu_para_sair`, construído a partir de `cIniciativaDemissao`
- As 5 bases de eventos da Aula 1 (`FtAbsenteismoMensalRH`, `FtAcidentesRH`, `FtHoraExtraRH`, `FtHorasIrregularesRH`, `FtMovimentoSalarialRH`)

**O que é novo hoje:**
- `datas_referencia.csv` — uma **data de referência por colaborador**. Ao construir qualquer métrica agregada, usem **apenas eventos até essa data** (inclusive). Isso não é opcional — é a mesma regra de janela temporal que vimos no material de apoio.

**Objetivo:** produzir uma única tabela, **uma linha por `nIdPessoa`**, juntando cadastro + alvo + métricas agregadas das 5 bases de eventos.

Dois exemplos abaixo já vêm resolvidos, como referência de padrão. As demais bases ficam para vocês.

## 0. Carregando os dados

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', None)

A2 = "./data/raw/"

cadastro = pd.read_csv(A2 + "FtFuncionarioRH_amostra.csv", sep=";")
alvo = pd.read_csv(A2 + "alvo_derivado.csv", sep=";")
datas_ref = pd.read_csv(A2 + "datas_referencia.csv", sep=";")
datas_ref['data_referencia'] = pd.to_datetime(datas_ref['data_referencia'])

print(cadastro.shape, alvo.shape, datas_ref.shape)
datas_ref.head()

(24000, 27) (24000, 2) (24000, 2)


,nIdPessoa,data_referencia
0,181076,2018-06-10
1,201073,2019-02-17
2,280267,2023-05-03
3,14686,2017-05-03
4,71442,2021-04-09


## 1. Função utilitária: filtrar eventos pela data de referência

Já está pronta — vocês vão usá-la em todas as bases.

In [2]:
cutoff_map = datas_ref.set_index('nIdPessoa')['data_referencia']

def filtra_por_data(df, col_data, dayfirst=True):
    """Mantem apenas linhas com data <= data de referencia do respectivo colaborador."""
    df = df.copy()
    df['__data'] = pd.to_datetime(df[col_data], errors='coerce', dayfirst=dayfirst)
    df['__ref'] = df['nIdPessoa'].map(cutoff_map)
    df = df[df['__data'].notna() & df['__ref'].notna() & (df['__data'] <= df['__ref'])]
    return df.drop(columns=['__data', '__ref'])

def to_num(s):
    return pd.to_numeric(s.astype(str).str.replace(",", "."), errors="coerce")

## 2. Exemplo resolvido — Acidentes

Coluna de data: `dDataAcidente`. Vamos gerar 3 métricas: número de eventos, quantos tiveram afastamento, e total de dias perdidos.

In [3]:
acid = pd.read_csv(A2 + "FtAcidentesRH.csv", sep=";")
acid = filtra_por_data(acid, 'dDataAcidente')

acid['nComAfastamento_n'] = to_num(acid['nComAfastamento'])
acid['nDiasPerdidos_n'] = to_num(acid['nDiasPerdidos'])
acid['data_evento'] = pd.to_datetime(acid['dDataAcidente'], dayfirst=True, errors='coerce')

agg_acid = acid.groupby('nIdPessoa').agg(
    acidentes_eventos=('nComAfastamento_n', 'count'),
    acidentes_com_afastamento=('nComAfastamento_n', 'sum'),
    acidentes_dias_perdidos=('nDiasPerdidos_n', 'sum'),
    ultimo_acidente=('data_evento', 'max'),
).reset_index()

print(agg_acid.shape)
agg_acid.head()

(1986, 5)


,nIdPessoa,acidentes_eventos,acidentes_com_afastamento,acidentes_dias_perdidos,ultimo_acidente
0,11,2,1,15,2020-09-01
1,128,1,1,4,2018-06-16
2,265,1,1,90,2020-07-16
3,676,1,1,30,2016-08-15
4,1105,1,1,21,2016-11-15


## 3. Exemplo resolvido — Movimentação Salarial

Coluna de data: `dMudanca`. Métricas: número de eventos, valor total, e **percentual médio** (média, não soma — percentual não se acumula da mesma forma que valor).

In [4]:
mov = pd.read_csv(A2 + "FtMovimentoSalarialRH.csv", sep=";")
mov = filtra_por_data(mov, 'dMudanca')

mov['nValor_n'] = to_num(mov['nValor'])
mov['nPerc_n'] = to_num(mov['nPerc'])
mov['data_evento'] = pd.to_datetime(mov['dMudanca'], dayfirst=True, errors='coerce')

agg_mov = mov.groupby('nIdPessoa').agg(
    mov_sal_eventos=('nValor_n', 'count'),
    mov_sal_valor_total=('nValor_n', 'sum'),
    mov_sal_perc_medio=('nPerc_n', 'mean'),
    ultimo_mov_sal=('data_evento', 'max'),
).reset_index()

print(agg_mov.shape)
agg_mov.head()

(14540, 5)


,nIdPessoa,mov_sal_eventos,mov_sal_valor_total,mov_sal_perc_medio,ultimo_mov_sal
0,11,8,838.003840,0.054387,2023-08-01
1,31,2,211.252214,0.050400,2017-08-01
2,57,6,558.894753,0.048567,2020-08-01
3,71,5,296.058198,0.039080,2020-08-01
4,73,3,262.274569,0.045633,2018-08-01


## 4. Exercício 1 — Absenteísmo

Coluna de data: `dAnoMes`. Colunas disponíveis: `nQtdeAbsenteismo`, `nHoraPrevista`, `cTipo`, `nIdPessoa`.

Construam uma tabela `agg_abse` com uma linha por `nIdPessoa` e as colunas:
- `abs_eventos` — quantidade de eventos
- `abs_qtd_total` — soma de `nQtdeAbsenteismo`
- `horas_previstas_total` — soma de `nHoraPrevista`

Sigam o mesmo padrão dos exemplos acima (filtrar por data, converter para número, agregar).

In [5]:
abse = pd.read_csv(A2 + "FtAbsenteismoMensalRH.csv", sep=";")

# dAnoMes vem com o mes abreviado em portugues ("ago 2017"), que o pd.to_datetime
# nao reconhece. Traduzimos as abreviacoes para o ingles antes de filtrar.
meses_pt_en = {'jan':'Jan','fev':'Feb','mar':'Mar','abr':'Apr','mai':'May',
               'jun':'Jun','jul':'Jul','ago':'Aug','set':'Sep','out':'Oct',
               'nov':'Nov','dez':'Dec'}
abse['dAnoMes'] = abse['dAnoMes'].astype(str).str.strip().replace(meses_pt_en, regex=True)

# filtrar por data de referencia (coluna dAnoMes)
abse = filtra_por_data(abse, 'dAnoMes')

# converter as colunas numericas relevantes (lembrem do to_num)
abse['nQtdeAbsenteismo_n'] = to_num(abse['nQtdeAbsenteismo'])
abse['nHoraPrevista_n'] = to_num(abse['nHoraPrevista'])
abse['data_evento'] = pd.to_datetime(abse['dAnoMes'], errors='coerce')

# agregar por nIdPessoa -> agg_abse (abs_eventos, abs_qtd_total, horas_previstas_total)
agg_abse = abse.groupby('nIdPessoa').agg(
    abs_eventos=('nIdPessoa', 'count'),            # cada linha = um dia registrado no ponto
    abs_qtd_total=('nQtdeAbsenteismo_n', 'sum'),   # minutos de ausencia (NaN = dia normal)
    horas_previstas_total=('nHoraPrevista_n', 'sum'),
    ultimo_absenteismo=('data_evento', 'max'),
).reset_index()

print(agg_abse.shape)
agg_abse.head()


(64, 5)


C:\Users\miguelaraujo-ieg\AppData\Local\Temp\ipykernel_31012\863360617.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['__data'] = pd.to_datetime(df[col_data], errors='coerce', dayfirst=dayfirst)
C:\Users\miguelaraujo-ieg\AppData\Local\Temp\ipykernel_31012\130426417.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  abse['data_evento'] = pd.to_datetime(abse['dAnoMes'], errors='coerce')


,nIdPessoa,abs_eventos,abs_qtd_total,horas_previstas_total,ultimo_absenteismo
0,11,2102,21217.0,919594.0,2023-08-01
1,31,386,1320.0,233619.0,2018-03-01
2,34,82,0.0,36000.0,2016-04-01
3,57,1309,2200.0,574161.0,2020-09-01
4,71,1331,25303.0,579480.0,2021-04-01


In [6]:
abse['dAnoMes'].value_counts()


dAnoMes
Feb 2016    730
Mar 2016    723
Jul 2016    699
Apr 2016    695
Aug 2016    663
           ... 
Jan 2024    223
Nov 2023    218
Oct 2022    213
Oct 2023    168
Feb 2024    153
Name: count, Length: 98, dtype: int64

## 5. Exercício 2 — Hora Extra

Coluna de data: `dAnoMes`. Colunas disponíveis: `nReferencia`, `nValor`, `nIdPessoa`.

Construam `agg_hext` com:
- `he_eventos` — quantidade de eventos
- `he_referencia_total` — soma de `nReferencia`
- `he_valor_total` — soma de `nValor`

In [7]:
hext = pd.read_csv(A2 + "FtHoraExtraRH.csv", sep=";")

# Mesma traducao de mes pt->en da base de absenteismo (ex.: "jul 2023" -> "Jul 2023")
meses_pt_en = {'jan':'Jan','fev':'Feb','mar':'Mar','abr':'Apr','mai':'May',
               'jun':'Jun','jul':'Jul','ago':'Aug','set':'Sep','out':'Oct',
               'nov':'Nov','dez':'Dec'}
hext['dAnoMes'] = hext['dAnoMes'].astype(str).str.strip().replace(meses_pt_en, regex=True)

# filtrar por data de referencia (coluna dAnoMes)
hext = filtra_por_data(hext, 'dAnoMes')

# converter as colunas numericas relevantes
hext['nReferencia_n'] = to_num(hext['nReferencia'])
hext['nValor_n'] = to_num(hext['nValor'])
hext['data_evento'] = pd.to_datetime(hext['dAnoMes'], errors='coerce')

# agregar por nIdPessoa -> agg_hext (he_eventos, he_referencia_total, he_valor_total)
agg_hext = hext.groupby('nIdPessoa').agg(
    he_eventos=('nReferencia_n', 'count'),
    he_referencia_total=('nReferencia_n', 'sum'),
    he_valor_total=('nValor_n', 'sum'),
    ultimo_hora_extra=('data_evento', 'max'),
).reset_index()

print(agg_hext.shape)
agg_hext.head()


C:\Users\miguelaraujo-ieg\AppData\Local\Temp\ipykernel_31012\863360617.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['__data'] = pd.to_datetime(df[col_data], errors='coerce', dayfirst=dayfirst)


(9721, 5)


C:\Users\miguelaraujo-ieg\AppData\Local\Temp\ipykernel_31012\1341081682.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  hext['data_evento'] = pd.to_datetime(hext['dAnoMes'], errors='coerce')


,nIdPessoa,he_eventos,he_referencia_total,he_valor_total,ultimo_hora_extra
0,11,2,14.88,378.86,2023-07-01
1,202,3,57.50,1216.13,2023-04-01
2,256,105,506.27,10514.70,2024-01-01
3,356,101,575.44,12077.01,2024-01-01
4,387,25,77.36,1341.95,2022-08-01


## 6. Exercício 3 — Horas Irregulares

Coluna de data: `dOcorrencia`. Colunas disponíveis: `nMinutosIrregularesExcedidos`, `nMinutosExtras`, `nIdPessoa`.

Construam `agg_hirr` com:
- `hi_eventos` — quantidade de eventos
- `hi_minutos_irregulares` — soma de `nMinutosIrregularesExcedidos`
- `hi_minutos_extras` — soma de `nMinutosExtras`

In [8]:
hirr = pd.read_csv(A2 + "FtHorasIrregularesRH.csv", sep=";", low_memory=False)

# filtrar por data de referencia (coluna dOcorrencia)
hirr = filtra_por_data(hirr, 'dOcorrencia')

# converter as colunas numericas relevantes
hirr['nMinutosIrregulares_n'] = to_num(hirr['nMinutosIrregularesExcedidos'])
hirr['nMinutosExtras_n'] = to_num(hirr['nMinutosExtras'])
hirr['data_evento'] = pd.to_datetime(hirr['dOcorrencia'], dayfirst=True, errors='coerce')

# agregar por nIdPessoa -> agg_hirr (hi_eventos, hi_minutos_irregulares, hi_minutos_extras)
agg_hirr = hirr.groupby('nIdPessoa').agg(
    hi_eventos=('nMinutosIrregulares_n', 'count'),
    hi_minutos_irregulares=('nMinutosIrregulares_n', 'sum'),
    hi_minutos_extras=('nMinutosExtras_n', 'sum'),
    ultimo_hora_irregular=('data_evento', 'max'),
).reset_index()

print(agg_hirr.shape)
agg_hirr.head()


(12391, 5)


,nIdPessoa,hi_eventos,hi_minutos_irregulares,hi_minutos_extras,ultimo_hora_irregular
0,11,2,893,0.0,2023-06-20
1,31,73,10438,0.0,2018-02-13
2,57,4,643,0.0,2020-07-28
3,71,2,391,0.0,2016-11-17
4,73,10,117,0.0,2019-01-18


## 7. Juntando tudo

Esta parte já está pronta — só roda depois que `agg_abse`, `agg_hext` e `agg_hirr` estiverem criados acima.

In [9]:
df = (cadastro
      .merge(alvo, on='nIdPessoa', how='left', validate='one_to_one')
      .merge(datas_ref, on='nIdPessoa', how='left', validate='one_to_one'))
for agg in [agg_abse, agg_acid, agg_hext, agg_hirr, agg_mov]:
    df = df.merge(agg, on='nIdPessoa', how='left')

event_cols = ['abs_eventos','abs_qtd_total','horas_previstas_total','acidentes_eventos',
    'acidentes_com_afastamento','acidentes_dias_perdidos','he_eventos','he_referencia_total',
    'he_valor_total','hi_eventos','hi_minutos_irregulares','hi_minutos_extras',
    'mov_sal_eventos','mov_sal_valor_total','mov_sal_perc_medio']
df[event_cols] = df[event_cols].fillna(0)
df['pediu_para_sair'] = df['pediu_para_sair'].astype(int)

print("Shape final:", df.shape)
df.head()

Shape final: (24000, 49)


,nIdPessoa,nCodColigada,cFuncao,dAnoMes,EMPRESA,nCodFilial,cMes,nAno,cCargo,cSituacao,cSecao,nNrDependentes,cEstadoCivil,cSexo,cCor,cEscolaridade,cEstadoEndereco,cCidadeEndereco,cTurno,nTempoDeCasaAnos,nIdade,cGeracaoNascimento,cFaixasTempoDeCasa,cFaixasIdade,nSalarioTotal,nRemuneracaoTotal,cPosicaoFaixaSalarial,pediu_para_sair,data_referencia,abs_eventos,abs_qtd_total,horas_previstas_total,ultimo_absenteismo,acidentes_eventos,acidentes_com_afastamento,acidentes_dias_perdidos,ultimo_acidente,he_eventos,he_referencia_total,he_valor_total,ultimo_hora_extra,hi_eventos,hi_minutos_irregulares,hi_minutos_extras,ultimo_hora_irregular,mov_sal_eventos,mov_sal_valor_total,mov_sal_perc_medio,ultimo_mov_sal
0,181076,47,Operador de Produção I,set 2018,UBERABA - ABATE AVES,142,set,2018,Operação - Produção,Demitido,Gr Aves - Frango Inteiro - Alimentar Máq Pacot...,NaN,Solteiro,Masculino,Parda,Do 6º ao 9º ano do ensino fundamental,MG,Uberaba,1.0,"0,24",33,Geração Y,Até 3 Meses,De 31 a 35,1005,1005,Abaixo de 80%,0,2018-06-10,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,0.0,0.00,0.00,NaT,0.0,0.0,0.0,NaT,0.0,0.000000,0.000000,NaT
1,201073,1,Ajudante de Serviços Gerais,mai 2019,CPG,4,mai,2019,Administração,Demitido,Limpeza Industrial - Higienização - 2º Turno,NaN,Solteiro,Feminino,Parda,Ensino médio completo,MS,Campo Grande,NaN,"0,24",32,Geração Y,Até 3 Meses,De 31 a 35,"1092,53","1092,53",Abaixo de 80%,0,2019-02-17,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,0.0,0.00,0.00,NaT,0.0,0.0,0.0,NaT,0.0,0.000000,0.000000,NaT
2,280267,47,Operador de Produção I,ago 2023,FORQUILHINHA - ABATE AVES,24,ago,2023,Operação - Produção,Demitido,Gr Aves - Escald/Depen/Evisc - Evisceração - 2...,NaN,Solteiro,Masculino,Parda,Ensino médio completo,SC,Forquilhinha,2.0,"2,55",29,Geração Y,De 2 a 5 Anos,De 26 a 30,"1843,53","1843,53",80%,1,2023-05-03,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,34.0,122.65,1904.61,2022-12-01,0.0,0.0,0.0,NaT,3.0,499.481131,0.111600,2022-10-01
3,14686,1,Operador de Produção,ago 2017,CGR,77,ago,2017,Operação - Produção,Demitido,Embalagem Secundária - Encaixotamento - 1º Turno,NaN,Solteiro,Masculino,Parda,Ensino médio incompleto,MS,Campo Grande,NaN,"1,55",25,Geração Y,De 1 a 2 Anos,De 19 a 25,"1032,43","1032,43",Abaixo de 80%,0,2017-05-03,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,0.0,0.00,0.00,NaT,0.0,0.0,0.0,NaT,2.0,119.302130,0.063450,2017-03-01
4,71442,47,Desossador de Coxa,jul 2021,FORQUILHINHA - ABATE AVES,24,jul,2021,Operação - Produção,Demitido,Gr Aves - Perna - Desossar Perna - 2° Turno,NaN,Solteiro,Masculino,Branca,Ensino fundamental completo,SC,Criciúma,2.0,"7,00",40,Geração X,De 5 a 10 Anos,De 36 a 45,"1637,81","1637,81",Abaixo de 80%,0,2021-04-09,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,49.0,76.32,1334.23,2021-04-01,3.0,1056.0,0.0,2019-12-04,6.0,315.951223,0.036467,2020-10-01


## 8. Conferência final

Antes de considerar pronto, respondam:
1. `df.shape[0]` bate com o número de colaboradores do cadastro?
2. A proporção de `pediu_para_sair` continua parecida com o que vimos na Aula 2?
3. Alguma das suas 3 colunas de `abs_eventos`, `he_eventos` ou `hi_eventos` parece grande ou pequena demais? Por quê?

In [10]:
print(df['pediu_para_sair'].value_counts(normalize=True))
print()
print(df[['abs_eventos','he_eventos','hi_eventos']].describe())

pediu_para_sair
1    0.513667
0    0.486333
Name: proportion, dtype: float64

        abs_eventos    he_eventos    hi_eventos
count  24000.000000  24000.000000  24000.000000
mean       1.664375     12.161458      8.950000
std       52.078696     26.417984     26.855846
min        0.000000      0.000000      0.000000
25%        0.000000      0.000000      0.000000
50%        0.000000      0.000000      1.000000
75%        0.000000     11.000000      6.000000
max     2262.000000    262.000000    714.000000


In [11]:
df

,nIdPessoa,nCodColigada,cFuncao,dAnoMes,EMPRESA,nCodFilial,cMes,nAno,cCargo,cSituacao,cSecao,nNrDependentes,cEstadoCivil,cSexo,cCor,cEscolaridade,cEstadoEndereco,cCidadeEndereco,cTurno,nTempoDeCasaAnos,nIdade,cGeracaoNascimento,cFaixasTempoDeCasa,cFaixasIdade,nSalarioTotal,nRemuneracaoTotal,cPosicaoFaixaSalarial,pediu_para_sair,data_referencia,abs_eventos,abs_qtd_total,horas_previstas_total,ultimo_absenteismo,acidentes_eventos,acidentes_com_afastamento,acidentes_dias_perdidos,ultimo_acidente,he_eventos,he_referencia_total,he_valor_total,ultimo_hora_extra,hi_eventos,hi_minutos_irregulares,hi_minutos_extras,ultimo_hora_irregular,mov_sal_eventos,mov_sal_valor_total,mov_sal_perc_medio,ultimo_mov_sal
0,181076,47,Operador de Produção I,set 2018,UBERABA - ABATE AVES,142,set,2018,Operação - Produção,Demitido,Gr Aves - Frango Inteiro - Alimentar Máq Pacot...,NaN,Solteiro,Masculino,Parda,Do 6º ao 9º ano do ensino fundamental,MG,Uberaba,1.0,"0,24",33,Geração Y,Até 3 Meses,De 31 a 35,1005,1005,Abaixo de 80%,0,2018-06-10,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,0.0,0.00,0.00,NaT,0.0,0.0,0.0,NaT,0.0,0.000000,0.000000,NaT
1,201073,1,Ajudante de Serviços Gerais,mai 2019,CPG,4,mai,2019,Administração,Demitido,Limpeza Industrial - Higienização - 2º Turno,NaN,Solteiro,Feminino,Parda,Ensino médio completo,MS,Campo Grande,NaN,"0,24",32,Geração Y,Até 3 Meses,De 31 a 35,"1092,53","1092,53",Abaixo de 80%,0,2019-02-17,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,0.0,0.00,0.00,NaT,0.0,0.0,0.0,NaT,0.0,0.000000,0.000000,NaT
2,280267,47,Operador de Produção I,ago 2023,FORQUILHINHA - ABATE AVES,24,ago,2023,Operação - Produção,Demitido,Gr Aves - Escald/Depen/Evisc - Evisceração - 2...,NaN,Solteiro,Masculino,Parda,Ensino médio completo,SC,Forquilhinha,2.0,"2,55",29,Geração Y,De 2 a 5 Anos,De 26 a 30,"1843,53","1843,53",80%,1,2023-05-03,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,34.0,122.65,1904.61,2022-12-01,0.0,0.0,0.0,NaT,3.0,499.481131,0.111600,2022-10-01
3,14686,1,Operador de Produção,ago 2017,CGR,77,ago,2017,Operação - Produção,Demitido,Embalagem Secundária - Encaixotamento - 1º Turno,NaN,Solteiro,Masculino,Parda,Ensino médio incompleto,MS,Campo Grande,NaN,"1,55",25,Geração Y,De 1 a 2 Anos,De 19 a 25,"1032,43","1032,43",Abaixo de 80%,0,2017-05-03,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,0.0,0.00,0.00,NaT,0.0,0.0,0.0,NaT,2.0,119.302130,0.063450,2017-03-01
4,71442,47,Desossador de Coxa,jul 2021,FORQUILHINHA - ABATE AVES,24,jul,2021,Operação - Produção,Demitido,Gr Aves - Perna - Desossar Perna - 2° Turno,NaN,Solteiro,Masculino,Branca,Ensino fundamental completo,SC,Criciúma,2.0,"7,00",40,Geração X,De 5 a 10 Anos,De 36 a 45,"1637,81","1637,81",Abaixo de 80%,0,2021-04-09,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,49.0,76.32,1334.23,2021-04-01,3.0,1056.0,0.0,2019-12-04,6.0,315.951223,0.036467,2020-10-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23995,259893,47,Operador de Produção I,out 2020,DOURADOS - ALIMENTOS PREPARADOS,61,out,2020,Operação - Produção,Demitido,Gr Industrializado - Hamburguer - Preparar Mas...,NaN,Casado,Masculino,Parda,Ensino fundamental completo,MS,Dourados,1.0,"0,12",25,Geração Y,Até 3 Meses,De 19 a 25,"1253,19","1253,19",Abaixo de 80%,1,2020-07-03,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,0.0,0.00,0.00,NaT,0.0,0.0,0.0,NaT,0.0,0.000000,0.000000,NaT
23996,149063,1,Operador de Produção,abr 2018,SPA,211,abr,2018,Operação - Produção,Demitido,Linha de Salga - Tombar Manta - 1º Turno,NaN,Solteiro,Masculino,Branca,Ensino fundamental completo,SP,Cajamar,NaN,"0,79",37,Geração X,De 6 Meses a 1 Ano,De 36 a 45,"1435,31","1435,31",Abaixo de 80%,0,2018-01-04,0.0,0.0,0.0,NaT,0.0,0.0,0.0,NaT,0.0,0.00,0.00,NaT,0.0,0.0,0.0,NaT,1.0,62.727041,0.045700,2017-06-19
23997,228814,1,Desossador II,nov 2020,CGR,77,nov,2020,Operação - Produção,Demitido,TR - Desossar Lagarto - 1º Turno,NaN,Casado,Masculino,Parda,Ensino médio completo,MS,Campo Grande,NaN,"1,02",25,Geração Y,De

---
# 9. Base cleaned para modelagem

Esta etapa gera uma linha por colaborador com o alvo, a data de referência e **somente eventos ocorridos até o corte individual**. Campos do cadastro não entram no arquivo cleaned: a fonte disponível possui snapshots posteriores ao corte em parte relevante da amostra e não pode ser usada para treinamento preditivo.

## Features entregues

| Grupo | Features | Construção |
|---|---|---|
| Acidentes | `acidentes_*`, `teve_acidente`, `dias_desde_ultimo_acidente` | Contagens/somas até o corte; presença e dias desde a última ocorrência. |
| Absenteísmo | `abs_*`, `taxa_absenteismo`, `teve_absenteismo`, `dias_desde_ultimo_absenteismo` | Contagens, total de ausência, jornada prevista e `abs_qtd_total / horas_previstas_total`. |
| Hora extra | `he_*`, `teve_hora_extra`, `dias_desde_ultima_hora_extra`, `he_valor_medio` | Ocorrências e valores até o corte; média calculada apenas quando há evento. |
| Jornada irregular | `hi_*`, `teve_hora_irregular`, `dias_desde_ultima_hora_irregular`, `taxa_hora_irregular` | Ocorrências/minutos até o corte e taxa por hora prevista. |
| Movimento salarial | `mov_sal_*`, `teve_mov_sal`, `dias_desde_ultimo_mov_sal`, `mov_sal_valor_medio` | Ocorrências e valores até o corte; média calculada apenas quando há evento. |

Para eventos inexistentes, os valores agregados são zero e `dias_desde_*` recebe `-1`; a flag `teve_*` preserva essa distinção. `nIdPessoa` fica somente para rastreabilidade e é excluído no treino.


In [12]:
# O cleaned seleciona explicitamente apenas dados permitidos no instante da previsão.
colunas_base = ['nIdPessoa', 'data_referencia', 'pediu_para_sair']
colunas_evento = [
    'acidentes_eventos', 'acidentes_com_afastamento', 'acidentes_dias_perdidos', 'ultimo_acidente',
    'abs_eventos', 'abs_qtd_total', 'horas_previstas_total', 'ultimo_absenteismo',
    'he_eventos', 'he_referencia_total', 'he_valor_total', 'ultimo_hora_extra',
    'hi_eventos', 'hi_minutos_irregulares', 'hi_minutos_extras', 'ultimo_hora_irregular',
    'mov_sal_eventos', 'mov_sal_valor_total', 'mov_sal_perc_medio', 'ultimo_mov_sal',
]
cleaned = df[colunas_base + colunas_evento].copy()

# Agregados de quantidade/valor: ausência de registro até o corte equivale a zero evento.
colunas_zero = [
    'acidentes_eventos', 'acidentes_com_afastamento', 'acidentes_dias_perdidos',
    'abs_eventos', 'abs_qtd_total', 'horas_previstas_total',
    'he_eventos', 'he_referencia_total', 'he_valor_total',
    'hi_eventos', 'hi_minutos_irregulares', 'hi_minutos_extras',
    'mov_sal_eventos', 'mov_sal_valor_total', 'mov_sal_perc_medio',
]
cleaned[colunas_zero] = cleaned[colunas_zero].fillna(0)

for ultima_data, flag, recencia in [
    ('ultimo_acidente', 'teve_acidente', 'dias_desde_ultimo_acidente'),
    ('ultimo_absenteismo', 'teve_absenteismo', 'dias_desde_ultimo_absenteismo'),
    ('ultimo_hora_extra', 'teve_hora_extra', 'dias_desde_ultima_hora_extra'),
    ('ultimo_hora_irregular', 'teve_hora_irregular', 'dias_desde_ultima_hora_irregular'),
    ('ultimo_mov_sal', 'teve_mov_sal', 'dias_desde_ultimo_mov_sal'),
]:
    cleaned[flag] = cleaned[ultima_data].notna().astype('int8')
    cleaned[recencia] = (cleaned['data_referencia'] - cleaned[ultima_data]).dt.days.clip(lower=0).fillna(-1).astype('int32')

cleaned['taxa_absenteismo'] = np.divide(
    cleaned['abs_qtd_total'], cleaned['horas_previstas_total'],
    out=np.zeros(len(cleaned), dtype=float),
    where=cleaned['horas_previstas_total'].to_numpy() > 0,
)
cleaned['taxa_hora_irregular'] = np.divide(
    cleaned['hi_minutos_irregulares'], cleaned['horas_previstas_total'],
    out=np.zeros(len(cleaned), dtype=float),
    where=cleaned['horas_previstas_total'].to_numpy() > 0,
)
cleaned['he_valor_medio'] = np.divide(
    cleaned['he_valor_total'], cleaned['he_eventos'], out=np.zeros(len(cleaned), dtype=float),
    where=cleaned['he_eventos'].to_numpy() > 0,
)
cleaned['mov_sal_valor_medio'] = np.divide(
    cleaned['mov_sal_valor_total'], cleaned['mov_sal_eventos'], out=np.zeros(len(cleaned), dtype=float),
    where=cleaned['mov_sal_eventos'].to_numpy() > 0,
)
cleaned = cleaned.drop(columns=['ultimo_acidente', 'ultimo_absenteismo', 'ultimo_hora_extra', 'ultimo_hora_irregular', 'ultimo_mov_sal'])

assert cleaned['nIdPessoa'].is_unique
assert cleaned['data_referencia'].notna().all()
assert cleaned['pediu_para_sair'].isin([0, 1]).all()
print(f'Cleaned: {cleaned.shape[0]:,} pessoas e {cleaned.shape[1] - 3} features seguras')
cleaned.head()


Cleaned: 24,000 pessoas e 29 features seguras


,nIdPessoa,data_referencia,pediu_para_sair,acidentes_eventos,acidentes_com_afastamento,acidentes_dias_perdidos,abs_eventos,abs_qtd_total,horas_previstas_total,he_eventos,he_referencia_total,he_valor_total,hi_eventos,hi_minutos_irregulares,hi_minutos_extras,mov_sal_eventos,mov_sal_valor_total,mov_sal_perc_medio,teve_acidente,dias_desde_ultimo_acidente,teve_absenteismo,dias_desde_ultimo_absenteismo,teve_hora_extra,dias_desde_ultima_hora_extra,teve_hora_irregular,dias_desde_ultima_hora_irregular,teve_mov_sal,dias_desde_ultimo_mov_sal,taxa_absenteismo,taxa_hora_irregular,he_valor_medio,mov_sal_valor_medio
0,181076,2018-06-10,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.000000,0.000000,0,-1,0,-1,0,-1,0,-1,0,-1,0.0,0.0,0.000000,0.000000
1,201073,2019-02-17,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.000000,0.000000,0,-1,0,-1,0,-1,0,-1,0,-1,0.0,0.0,0.000000,0.000000
2,280267,2023-05-03,1,0.0,0.0,0.0,0.0,0.0,0.0,34.0,122.65,1904.61,0.0,0.0,0.0,3.0,499.481131,0.111600,0,-1,0,-1,1,153,0,-1,1,214,0.0,0.0,56.017941,166.493710
3,14686,2017-05-03,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,2.0,119.302130,0.063450,0,-1,0,-1,0,-1,0,-1,1,63,0.0,0.0,0.000000,59.651065
4,71442,2021-04-09,0,0.0,0.0,0.0,0.0,0.0,0.0,49.0,76.32,1334.23,3.0,1056.0,0.0,6.0,315.951223,0.036467,0,-1,0,-1,1,8,1,492,1,190,0.0,0.0,27.229184,52.658537


## 9.1 Persistência da camada cleaned

O arquivo CSV mantém os tipos de dados e é a única entrada permitida para o notebook de treinamento. Isso separa a engenharia da modelagem e cria uma fronteira explícita contra leakage cadastral.


In [13]:
caminho_cleaned = Path('data/cleaned/turnover_features.csv')
caminho_cleaned.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(caminho_cleaned, index=False, date_format='%Y-%m-%d')
print(f'Base cleaned salva: {caminho_cleaned.resolve()}')
print(f'Linhas: {len(cleaned):,} | Positivos: {cleaned.pediu_para_sair.mean():.1%}')


Base cleaned salva: C:\Users\miguelaraujo-ieg\OneDrive - Instituto Germinare\3°ANO\Ciencia de Dados 2\Turnover\turnover\data\cleaned\turnover_features.csv
Linhas: 24,000 | Positivos: 51.4%
